In [9]:
import os
import pandas as pd
import struct
import matplotlib.pyplot as plt

root_dir=os.path.join('x:/Marphys_Archive','Data','OSNAP','RTEB1L-FetchAZA')
incsv=os.path.join(root_dir,"pressure-record-20230907.csv")
da=pd.read_csv(incsv)

ds=da.set_index('Record Time')
ds=ds[["Pressure Digiquartz (kPa)","Pressure Presens (kPa)"]]
ds=ds.loc['2022/07/17 08:00':'2023/09/05 08:00']
pres_data=ds['Pressure Digiquartz (kPa)']
pres_data

Record Time
2022/07/17 08:00:00    18284.66211
2022/07/17 09:00:00    18285.40820
2022/07/17 10:00:00    18282.41602
2022/07/17 11:00:00    18276.63672
2022/07/17 12:00:00    18269.76563
                          ...     
2023/09/05 03:00:00    18260.98633
2023/09/05 04:00:00    18261.77148
2023/09/05 05:00:00    18265.47461
2023/09/05 06:00:00    18271.05664
2023/09/05 07:00:00    18276.91602
Name: Pressure Digiquartz (kPa), Length: 9958, dtype: float64

In [34]:
da

,Unnamed: 0,Record Time,Pressure Presens (kPa),Serial Number_x,Pressure Digiquartz (kPa),Serial Number_y
0,0,2022/05/10 14:00:00,100.09644,862600094,101.24680,149405
1,1,2022/05/10 14:15:00,105.67796,862600094,103.54306,149405
2,2,2022/05/10 14:30:00,105.93211,862600094,103.86324,149405
3,3,2022/05/10 14:45:00,105.89406,862600094,107.37601,149405
4,4,2022/05/10 15:15:00,105.91891,862600094,107.40330,149405
...,...,...,...,...,...,...
10023,10023,2023/09/05 07:00:00,18274.67188,862600094,18276.91602,149405
10024,10024,2023/09/05 08:00:00,18279.55078,862600094,18281.75391,149405
10025,10025,2023/09/05 09:00:00,18282.29297,862600094,18284.51953,149405
10026,10026,2023/09/05 10:00:00,18282.21680,862600094,18284.33203,149405


In [31]:
import numpy as np
from scipy.signal import butter,filtfilt
# Filter requirements.
fs = 1/2       # sample rate, Hz
cutoff = 1/48      # desired cutoff frequency of the filter, Hz ,      slightly higher than actual 1.2 Hz
nyq = 0.5 * fs  # Nyquist Frequency
order = 6 # filter order, default = 6  (for butterworth filter)

def butter_lowpass_filter(data, cutoff, fs, order):
    normal_cutoff = cutoff / nyq
    # Get the filter coefficients 
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

y = butter_lowpass_filter(pres_data, cutoff, fs, order)

In [33]:
fig, ax = plt.subplots(figsize=(18, 9))
yy=pres_data.values
xx=ds.index.values
xx=pd.to_datetime(xx)
ax.plot(xx,yy,"-",
              color="blue",
              linewidth=0.2,
              label="Raw data")
ax.plot(xx,y,"-",
              color="red",
              linewidth=0.5,
              label="48 hour low pass filter")
plt.xticks(fontsize=18, rotation=45)
plt.yticks(fontsize=18)
plt.ylabel("Decibar",fontsize=18)
plt.grid()
plt.title("Fetch BPR Pressure record", fontsize=20)
plt.legend(fontsize=18)
#save the figure
figfile = os.path.join('Fetch-bpr')
plt.savefig(figfile)  
plt.close()
